In [2]:
import sys
print(sys.executable)
!{sys.executable} -m pip install earthengine-api

c:\Users\user\.pyenv\pyenv-win\versions\3.12.2\python.exe
  Using cached earthengine_api-1.7.22-py3-none-any.whl.metadata (2.2 kB)
  Using cached google_cloud_storage-3.10.1-py3-none-any.whl.metadata (15 kB)
  Using cached google_api_python_client-2.194.0-py3-none-any.whl.metadata (7.0 kB)
  Using cached google_auth-2.49.2-py3-none-any.whl.metadata (6.2 kB)
  Using cached google_auth_httplib2-0.3.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached httplib2-0.31.2-py3-none-any.whl.metadata (2.2 kB)
  Using cached google_api_core-2.30.3-py3-none-any.whl.metadata (3.1 kB)
  Using cached uritemplate-4.2.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached google_cloud_core-2.5.1-py3-none-any.whl.metadata (2.8 kB)
  Using cached google_resumable_media-2.8.2-py3-none-any.whl.metadata (2.2 kB)
  Using cached googleapis_common_protos-1.74.0-py3-none-any.whl.metadata (9.2 kB)
  Using cached protobuf-7.34.1-cp310-abi3-win_amd6

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: c:\Users\user\.pyenv\pyenv-win\versions\3.12.2\python.exe -m pip install --upgrade pip


In [ ]:
import ee
import time
import traceback

# ==========================
# 1. AUTHENTICATION & INITIALIZATION
# ==========================

ee.Authenticate(force=True)
ee.Initialize()

# ==========================
# 2. CONFIGURATION
# ==========================
DRIVE_FOLDER = 'NEW_MASKS_EG' \
'' \
'' \
'' \
''
START_DATE = '2026-01-01'
END_DATE = '2026-04-01'

SAMPLES_PER_CLASS = {
    1: 1000,
    2: 1000,
    3: 1000,
    4: 1000,
}

# Per-class filename offsets to avoid overwriting already downloaded files
START_INDEX_OFFSET = {
    1: 0,
    2: 0,
    3: 0,
    4: 0,
}

PATCH_SIZE_METERS = 1280   # Result is 256x256 at 10m scale
DOMINANCE_THRESHOLD = 0.6
POINT_POOL_MULTIPLIER = 5  # Larger pool because many candidates fail class-dominance check
QUEUE_FULL_SLEEP_SECONDS = 30 * 60

# Class Definitions (West, South, East, North)
class_regions = {
    1: [ee.Geometry.Rectangle([30.0, 30.2, 31.8, 31.5]), ee.Geometry.Rectangle([30.7, 27.0, 31.2, 28.5])], # Greenery
    2: [ee.Geometry.Rectangle([25.5, 26.0, 28.5, 28.0]), ee.Geometry.Rectangle([32.5, 24.5, 34.0, 26.0])], # Sand
    3: [ee.Geometry.Rectangle([32.5, 22.5, 33.2, 23.8]), ee.Geometry.Rectangle([34.2, 25.0, 35.5, 27.0])], # Water
    4: [ee.Geometry.Rectangle([31.15, 29.95, 31.35, 30.15]), ee.Geometry.Rectangle([29.85, 31.15, 30.05, 31.25])] # Cement
}

# ==========================
# 3. PROCESSING FUNCTIONS
# ==========================

def mask_clouds(img):
    scl = img.select('SCL')
    # Keep only clear, water, and non-vegetated pixels (remove clouds, shadows, snow)
    mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    return img.updateMask(mask)

def remap_dw(dw_img):
    """Maps Dynamic World labels to your specific 1-4 classification."""
    # Original DW: 0:Water, 1:Trees, 2:Grass, 3:Flooded, 4:Crops, 5:Shrub, 6:Built, 7:Bare, 8:Snow
    # Target: 0:unknown, 1:Greenery, 2:Sand, 3:Water, 4:Cement
    return dw_img.remap([0,1,2,3,4,5,6,7,8],
                        [3,1,1,0,1,0,4,2,0]).rename('mask').toByte()

def is_valid_patch(label_img, target_class, geometry):
    """Checks if the patch is dominated by the target class."""
    stats = label_img.reduceRegion(
        reducer=ee.Reducer.frequencyHistogram(),
        geometry=geometry,
        scale=10,
        maxPixels=1e8
    )
    hist = ee.Dictionary(stats.get('mask'))
    total = ee.Number(hist.values().reduce(ee.Reducer.sum()))
    target_count = ee.Number(hist.get(str(target_class), 0))
    return target_count.divide(total).gt(DOMINANCE_THRESHOLD)

def is_queue_full_error(exc):
    return 'Too many tasks already in the queue' in str(exc)

# ==========================
# 4. MAIN EXECUTION LOOP (ROUND ROBIN)
# ==========================

class_ids = list(class_regions.keys())

# Per-class runtime state
class_state = {}
for cls_id in class_ids:
    target = SAMPLES_PER_CLASS.get(cls_id, 0)
    combined_region = ee.FeatureCollection([ee.Feature(r) for r in class_regions[cls_id]]).geometry()
    point_pool_size = max(target * POINT_POOL_MULTIPLIER, 1)
    points_list = ee.FeatureCollection.randomPoints(combined_region, point_pool_size).toList(point_pool_size)

    class_state[cls_id] = {
        'target': target,
        'success': 0,
        'attempt': 0,
        'point_pool_size': point_pool_size,
        'points_list': points_list,
        'exhausted': target == 0,
    }

print("\n--- ROUND-ROBIN COLLECTION STARTED ---")
for cls_id in class_ids:
    print(f"Class {cls_id}: target={class_state[cls_id]['target']} offset={START_INDEX_OFFSET.get(cls_id, 0)}")

total_tasks_submitted = 0

while True:
    # Stop when all classes are done/exhausted.
    if all(
        (state['success'] >= state['target']) or state['exhausted']
        for state in class_state.values()
    ):
        break

    for cls_id in class_ids:
        state = class_state[cls_id]

        if state['success'] >= state['target'] or state['exhausted']:
            continue

        try:
            if state['attempt'] >= state['point_pool_size']:
                state['exhausted'] = True
                print(f"Class {cls_id}: point pool exhausted at {state['success']}/{state['target']}.")
                continue

            # Use a local attempt index so queue-full retries can re-use the same candidate.
            attempt_idx = state['attempt']
            pt = ee.Feature(state['points_list'].get(attempt_idx)).geometry()
            patch = pt.buffer(PATCH_SIZE_METERS / 2).bounds()

            # Process Imagery: Sentinel-2 SR
            s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                  .filterBounds(patch)
                  .filterDate(START_DATE, END_DATE)
                  .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
                  .map(mask_clouds)
                  # Essential bands for ML (10m and 20m resolution)
                  .select(['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12'])
                  .median()
                  .clip(patch))

            # Keep only original Sentinel-2 bands and convert to Float32
            final_img = s2.toFloat()

            # Process Labels: Dynamic World
            label = remap_dw(
                ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1')
                  .filterBounds(patch)
                  .filterDate(START_DATE, END_DATE)
                  .select('label')
                  .mode()
                  .clip(patch)
            )

            # Verification check
            if is_valid_patch(label, cls_id, patch).getInfo():
                current_index = START_INDEX_OFFSET.get(cls_id, 0) + state['success']
                file_id = f'cls{cls_id}_s{current_index}'

                img_task = ee.batch.Export.image.toDrive(
                    image=final_img,
                    description=f'{file_id}_img',
                    folder=DRIVE_FOLDER,
                    region=patch,
                    dimensions="256x256",
                    fileFormat='GeoTIFF'
                )

                msk_task = ee.batch.Export.image.toDrive(
                    image=label,
                    description=f'{file_id}_msk',
                    folder=DRIVE_FOLDER,
                    region=patch,
                    dimensions="256x256",
                    fileFormat='GeoTIFF'
                )

                img_started = False
                try:
                    img_task.start()
                    img_started = True
                    msk_task.start()
                except Exception as export_exc:
                    if is_queue_full_error(export_exc):
                        if img_started:
                            try:
                                img_task.cancel()
                                print(f"Queue full while starting mask for {file_id}; canceled image task to avoid orphan image.")
                            except Exception as cancel_exc:
                                print(f"Queue full after image start for {file_id}, and cancel failed: {cancel_exc}")
                        print(f"Queue full. Sleeping for {QUEUE_FULL_SLEEP_SECONDS // 60} minutes before retry...")
                        time.sleep(QUEUE_FULL_SLEEP_SECONDS)
                        continue
                    raise

                state['success'] += 1
                total_tasks_submitted += 2  # One for image, one for mask

                if state['success'] % 20 == 0:
                    print(f"Class {cls_id}: {state['success']}/{state['target']} submitted")

            # Candidate consumed (valid or invalid), advance attempt index.
            state['attempt'] += 1

        except Exception as e:
            if is_queue_full_error(e):
                print(f"Queue full while processing class {cls_id} attempt {state['attempt']}; sleeping for {QUEUE_FULL_SLEEP_SECONDS // 60} minutes...")
                time.sleep(QUEUE_FULL_SLEEP_SECONDS)
                continue

            print(f"Error processing class {cls_id} at attempt {state['attempt']}:")
            traceback.print_exc()
            state['attempt'] += 1
            # Simple back-off to prevent hitting GEE request rate limits
            time.sleep(0.5)
            continue

print("\nAll batches submitted. Check https://code.earthengine.google.com/tasks to monitor progress.")
for cls_id in class_ids:
    s = class_state[cls_id]
    print(f"Class {cls_id}: submitted {s['success']}/{s['target']} (attempted {s['attempt']})")

EEException: Cannot authenticate: Invalid request.